In [1]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("MyFirstSparkApp").master("local[*]").getOrCreate()

sc = spark.sparkContext

26/09/06 15:39:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/06 15:39:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/06 15:39:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/06 15:39:26 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/09/06 15:39:26 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [3]:
SparkSession.builder

In [4]:
rdd = sc.textFile("payments_employee.csv")
rdd_header = rdd.first()
print(rdd_header)

First Name,Last Name,Email,Phone,Gender,Age,Job Title,Years Of Experience,Salary,Department


In [5]:
rdd_filtered = rdd.filter(lambda row: row != rdd_header)
rdd_filtered

PythonRDD[3] at RDD at PythonRDD.scala:53

In [6]:
SALARY_INDEX = 8

def filter_salary(line):
    try:
        fields = line.split(',')
        salary_text = fields[SALARY_INDEX].strip().replace('"','')
        salary = float(salary_text)
        return salary > 15500
    except (ValueError, IndexError):
        return False
    
filtered_employees_rdd = rdd_filtered.filter(filter_salary)

employee_count = filtered_employees_rdd.count()
print(f"Number of employees with salary greather than 15500: {employee_count}")

Number of employees with salary greather than 15500: 91455


In [7]:
def extract_salary(line):
    try:
        fields=line.split(',')
        return float(fields[SALARY_INDEX].strip().replace('"',''))
    except (ValueError, IndexError):
        return 0.0
    
salary_rdd = rdd_filtered.map(extract_salary)
max_salary = salary_rdd.max()

print(f"Max salary of all employees: {max_salary}")

max_sal = rdd_filtered.map(lambda line:float(line.split(',')[SALARY_INDEX].strip().replace('"',''))).max()
print(f"max salary in single line: {max_sal}")

Max salary of all employees: 150000.0
max salary in single line: 150000.0


In [30]:
#department has the highest average salary among senior employees (Years Of Experience ≥ 10). 
# Output only the department name as a single value
DEPT_INDEX=9
SALARY_INDEX=8
EXP_INDEX=7
senior_data = rdd_filtered.filter(lambda line: int(float(line.split(',')[EXP_INDEX].strip().replace('"',''))) >= 10)
senior_count = senior_data.count()
print(f"Number seniors count: {senior_count}")


dept_sal_map = senior_data.map(lambda line: (line.split(',')[DEPT_INDEX].strip().replace('"',''),
                                                   (float(line.split(',')[SALARY_INDEX].strip().replace('"','')),1)
                                               ))
dept_sal_map_count = dept_sal_map.count()
print(f"Number of dept sal map count: {dept_sal_map_count}")

dept_totals = dept_sal_map.reduceByKey(lambda a,b:(a[0]+b[0], a[1]+b[1]))
dept_totals_count = dept_totals.count()
print(f"Number of dept totals:{dept_totals_count}")
for dept,tot_sal in dept_totals.collect():
    print(f"Department - {dept} with total salary- ${tot_sal}")
    
dept_avg = dept_totals.mapValues(lambda total_n_count: total_n_count[0]/total_n_count[1])
for dept,avg_sal in dept_avg.collect():
    print(f"Department: '{dept}' with avg salary: ${avg_sal:.2f}")

highest_avg_dept = dept_avg.max(key=lambda x:x[1])

print(f"Department: '{highest_avg_dept[0]}' has Highest average salary: ${highest_avg_dept[1]:.2f}")

Number seniors count: 52726


Number of dept sal map count: 52726


Number of dept totals:8
Department - Analytics with total salary- $(497012289.0, 6515)
Department - Marketing with total salary- $(486533327.0, 6426)
Department - HR with total salary- $(508826185.0, 6682)
Department - Sales with total salary- $(501529437.0, 6539)
Department - IT with total salary- $(507793416.0, 6675)
Department - Product with total salary- $(508501018.0, 6570)
Department - Finance with total salary- $(513022044.0, 6701)
Department - Operations with total salary- $(505067130.0, 6618)
Sales - is the Department with total salary payout as the highest (501529437.0, 6539)
Department: 'Analytics' with avg salary: $76287.38
Department: 'Marketing' with avg salary: $75713.25
Department: 'HR' with avg salary: $76148.79
Department: 'Sales' with avg salary: $76698.19
Department: 'IT' with avg salary: $76073.92
Department: 'Product' with avg salary: $77397.42
Department: 'Finance' with avg salary: $76559.03
Department: 'Operations' with avg salary: $76317.18
Department: 'Product

In [43]:
# Find the department that has the highest proportion of senior employees (Years Of Experience > 10).
dept_senior_flag = rdd_filtered.map(lambda line: (
                                                line.split(',')[DEPT_INDEX].strip().replace('"',''),
                                                     (
                                                    1 if float(line.split(',')[EXP_INDEX].strip().replace('"','')) > 10 else 0,#senior flag
                                                    1 # Total count flag
                                                     )
                                                 )
                                   )
dept_counts = dept_senior_flag.reduceByKey(lambda a,b:(a[0]+b[0], a[1]+b[1]))
    
dept_proportion = dept_counts.mapValues(lambda counts:counts[0]/counts[1])
print(dept_proportion.first())

highest_proportion = dept_proportion.max(key=lambda x:x[1])
print(f"Department with highest proportion of senior employees: {highest_proportion[0]}")
print(f"Proportion: {highest_proportion[1]:.2%} of their workforce")


('Analytics', 0.473518234009865)
Department with highest proportion of senior employees: Finance
Proportion: 48.52% of their workforce


In [32]:
dept_salaries = rdd_filtered.map(lambda line: (
                                    line.split(',')[DEPT_INDEX].strip().replace('"',''),
                                    float(line.split(',')[SALARY_INDEX].strip().replace('"',''))
                                    )
                                )
dept_totals = dept_salaries.reduceByKey(lambda a,b:a+b)
dept_max_tot_sal = dept_totals.max(key=lambda x:x[1])
# Find the department whose total salary payout is the highest
print(f"{dept_max_tot_sal[0]} - is the Department with total salary payout as the highest {dept_max_tot_sal[1]}")


HR - is the Department with total salary payout as the highest 965179948.0


In [37]:
# Identify the single employee with the highest years of experience in the organization
sorted_data = rdd_filtered.sortBy(lambda x: float(x.split(',')[EXP_INDEX].strip().replace('"','')), ascending=False)
most_senior_emp = sorted_data.first()
print(most_senior_emp)

Pooja,Singh,pooja.singh11@email.com,9000000011,Male,32,Data Analyst,25,24574,Analytics


In [ ]:
# Count total employees across all departments whose salary is above the median salary of their department.

